# STAT 764 · Capstone Project 

Jake

Caleb


Cole

In [1]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
found = ([p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists()]
         + [c.parent.parent for c in here.glob("*/course/stat764.py")])
if os.environ.get("STAT764_REPO"):          # your clone, when it is not above you
    found.insert(0, pathlib.Path(os.environ["STAT764_REPO"]))

if found:                       # you are inside (or just outside) your clone
    sys.path.insert(0, str(found[0] / "course"))
else:                           # Colab, or a copy saved outside the clone
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/DataScienceUWL/stat764-fall2026"
        "/main/course/stat764.py", "stat764.py")
    sys.path.insert(0, ".")
    print("  (no local clone found — pulled the helpers from GitHub)")

from stat764 import load

In [2]:
import numpy as np
import pandas as pd

# Load the TidyTuesday hotels dataset directly into a Pandas DataFrame
hotels = pd.read_csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2020/2020-02-11/hotels.csv')

# Verify it loaded correctly
print(hotels.head())
print("Shape:", hotels.shape)

          hotel  is_canceled  lead_time  arrival_date_year arrival_date_month  \
0  Resort Hotel            0        342               2015               July   
1  Resort Hotel            0        737               2015               July   
2  Resort Hotel            0          7               2015               July   
3  Resort Hotel            0         13               2015               July   
4  Resort Hotel            0         14               2015               July   

   arrival_date_week_number  arrival_date_day_of_month  \
0                        27                          1   
1                        27                          1   
2                        27                          1   
3                        27                          1   
4                        27                          1   

   stays_in_weekend_nights  stays_in_week_nights  adults  ...  deposit_type  \
0                        0                     0       2  ...    No Deposit   
1     

In [3]:
# Cancellation rate across all rows (dataset column is spelled is_canceled).
canceled = hotels["is_canceled"].eq(1)
print(f"Cancellation rate: {canceled.mean():.2%} ({canceled.sum():,}/{len(hotels):,} rows)")

# Count missing values for every variable, including columns with zero missing values.
print("\nMissing values by variable:")
print(hotels.isna().sum().to_string())

Cancellation rate: 37.04% (44,224/119,390 rows)

Missing values by variable:
hotel                                  0
is_canceled                            0
lead_time                              0
arrival_date_year                      0
arrival_date_month                     0
arrival_date_week_number               0
arrival_date_day_of_month              0
stays_in_weekend_nights                0
stays_in_week_nights                   0
adults                                 0
children                               4
babies                                 0
meal                                   0
country                              488
market_segment                         0
distribution_channel                   0
is_repeated_guest                      0
previous_cancellations                 0
previous_bookings_not_canceled         0
reserved_room_type                     0
assigned_room_type                     0
booking_changes                        0
deposit_type         

In [4]:
# Libraries
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, cross_val_score, train_test_split

In [5]:
# Count exact row patterns, from most to least frequent
row_counts = (
    hotels.groupby(hotels.columns.tolist(), dropna=False, sort=False)
    .size()
    .sort_values(ascending=False)
)

print("Most repeated exact rows:")
print(row_counts.head(10).to_string())
print(f"\nRows beyond the first copy: {hotels.duplicated().sum():,}")

Most repeated exact rows:
hotel       is_canceled  lead_time  arrival_date_year  arrival_date_month  arrival_date_week_number  arrival_date_day_of_month  stays_in_weekend_nights  stays_in_week_nights  adults  children  babies  meal  country  market_segment  distribution_channel  is_repeated_guest  previous_cancellations  previous_bookings_not_canceled  reserved_room_type  assigned_room_type  booking_changes  deposit_type  agent  company  days_in_waiting_list  customer_type  adr    required_car_parking_spaces  total_of_special_requests  reservation_status  reservation_status_date
City Hotel  1            277        2016               November            46                        7                          1                        2                     2       0.0       0       BB    PRT      Groups          TA/TO                 0                  0                       0                               A                   A                   0                Non Refund    NaN    NaN    

# Removing Duplicated Rows

In [6]:
print(f"Rows belonging to duplicate groups: {hotels.duplicated(keep=False).sum():,}")
before = len(hotels)
hotels = hotels.drop_duplicates().reset_index(drop=True)
print(f"Removed {before - len(hotels):,} duplicate rows; {len(hotels):,} rows remain.")

Rows belonging to duplicate groups: 40,165
Removed 31,994 duplicate rows; 87,396 rows remain.


# Addressing Leaks

In [7]:
leakage_cols = ["reservation_status", "reservation_status_date", "deposit_type"]
# Removing these columns because they are not known at the time of booking and would leak information about cancellations.
hotels = hotels.drop(columns=leakage_cols)

In [8]:
# Cancellation rate across all rows (dataset column is spelled is_canceled).
canceled = hotels["is_canceled"].eq(1)
print(f"Cancellation rate: {canceled.mean():.2%} ({canceled.sum():,}/{len(hotels):,} rows)")

# Count missing values for every variable, including columns with zero missing values.
print("\nMissing values by variable:")
print(hotels.isna().sum().to_string())

Cancellation rate: 27.49% (24,025/87,396 rows)

Missing values by variable:
hotel                                 0
is_canceled                           0
lead_time                             0
arrival_date_year                     0
arrival_date_month                    0
arrival_date_week_number              0
arrival_date_day_of_month             0
stays_in_weekend_nights               0
stays_in_week_nights                  0
adults                                0
children                              4
babies                                0
meal                                  0
country                             452
market_segment                        0
distribution_channel                  0
is_repeated_guest                     0
previous_cancellations                0
previous_bookings_not_canceled        0
reserved_room_type                    0
assigned_room_type                    0
booking_changes                       0
agent                             12193
comp

# Splitting the Data

- We created an arrival data column that combines the arrival date information into one column

In [9]:
arrival_date = pd.to_datetime(
    hotels["arrival_date_year"].astype(str)
    + "-" + hotels["arrival_date_month"]
    + "-" + hotels["arrival_date_day_of_month"].astype(str),
    format="%Y-%B-%d",
)

# Split by arrival date; estimate booking date as arrival date minus lead_time.
cutoff = arrival_date.max().normalize() - pd.DateOffset(months=5)
booking_date = arrival_date - pd.to_timedelta(hotels["lead_time"], unit="D")

is_holdout = arrival_date >= cutoff
# This is our training data raw
train = hotels.loc[~is_holdout].copy()
# This is our test set raw, which we will use for holdout evaluation
holdout = hotels.loc[is_holdout].copy()

# Holdout observations booked before the holdout cutoff are cross-period records.
cross_period = holdout.loc[booking_date.loc[holdout.index] < cutoff]
print(f"Holdout starts: {cutoff.date()}")
print(f"Training rows: {len(train):,}; holdout rows: {len(holdout):,}")
print(
    f"Cross-period rows excluded from training: {len(cross_period):,} "
    "(their booking date is before the cutoff, but arrival is in the holdout)"
)

# Ten chronological folds using the training data's arrival dates.
train_arrivals = arrival_date.loc[train.index]
edges = pd.date_range(train_arrivals.min(), train_arrivals.max(), periods=11)
fold_id = pd.cut(train_arrivals, bins=edges, labels=False, include_lowest=True)

time_folds = [
    train.loc[fold_id.eq(i)].copy()
    for i in range(10)
]

for i, fold in enumerate(time_folds, start=1):
    fold_dates = arrival_date.loc[fold.index]
    print(
        f"Fold {i}: {fold_dates.min().date()} to {fold_dates.max().date()} "
        f"({len(fold):,} rows)"
    )

# Locate cross-period records' booking dates within the training-fold date ranges.
cross_booking_folds = pd.cut(
    booking_date.loc[cross_period.index],
    bins=edges,
    labels=False,
    include_lowest=True,
)
counts_by_fold = (
    cross_booking_folds.value_counts()
    .reindex(range(10), fill_value=0)
)

print("\nCross-period booking dates falling in each training-fold date range:")
for i, count in counts_by_fold.items():
    print(f"Fold {i + 1}: {count:,}")
print(f"Booking dates outside those fold date ranges: {cross_booking_folds.isna().sum():,}")

Holdout starts: 2017-03-31
Training rows: 65,367; holdout rows: 22,029
Cross-period rows excluded from training: 13,240 (their booking date is before the cutoff, but arrival is in the holdout)
Fold 1: 2015-07-01 to 2015-09-02 (4,295 rows)
Fold 2: 2015-09-03 to 2015-11-05 (5,645 rows)
Fold 3: 2015-11-06 to 2016-01-08 (3,828 rows)
Fold 4: 2016-01-09 to 2016-03-12 (5,597 rows)
Fold 5: 2016-03-13 to 2016-05-15 (7,995 rows)
Fold 6: 2016-05-16 to 2016-07-17 (7,631 rows)
Fold 7: 2016-07-18 to 2016-09-19 (8,650 rows)
Fold 8: 2016-09-20 to 2016-11-22 (8,170 rows)
Fold 9: 2016-11-23 to 2017-01-25 (6,177 rows)
Fold 10: 2017-01-26 to 2017-03-30 (7,379 rows)

Cross-period booking dates falling in each training-fold date range:
Fold 1: 0
Fold 2: 0
Fold 3: 0
Fold 4: 3
Fold 5: 194
Fold 6: 392
Fold 7: 1,078
Fold 8: 2,125
Fold 9: 3,861
Fold 10: 5,587
Booking dates outside those fold date ranges: 0


In [10]:
for column in ["agent", "company"]:
    print(f"\nCounts for {column} (including missing values):")
    print(hotels[column].value_counts(dropna=False).to_string())


Counts for agent (including missing values):
agent
9.0      28759
240.0    13028
NaN      12193
14.0      3349
7.0       3300
250.0     2779
241.0     1644
28.0      1502
8.0       1383
1.0       1232
6.0       1117
40.0       986
314.0      844
242.0      722
83.0       614
85.0       524
243.0      477
171.0      402
27.0       395
3.0        363
22.0       345
11.0       318
15.0       303
196.0      281
177.0      277
96.0       272
138.0      257
37.0       225
16.0       220
229.0      218
5.0        216
10.0       214
21.0       199
42.0       194
115.0      192
156.0      187
26.0       187
175.0      182
195.0      181
86.0       180
273.0      179
251.0      175
134.0      168
143.0      163
298.0      160
152.0      156
168.0      153
315.0      151
19.0       151
12.0       133
2.0        129
147.0      128
95.0       125
410.0      121
20.0       117
146.0      112
142.0      111
30.0       106
94.0       103
330.0       99
89.0        90
29.0        84
52.0        82
191

- Changing all of the NAN in `company` and `agent` to "none"

In [11]:
hotels[["agent", "company"]] = hotels[["agent", "company"]].fillna("none")

# STEP 2: Pipeline

In [12]:
# X_train and y_train are the features and target variable for the training dataset, respectively. 
# The target variable is "is_canceled", which indicates whether a booking was canceled (1) or not (0). 
# The features are all other columns in the training dataset except for "is_canceled".
X_train = train.drop(columns=["is_canceled"])
y_train = train["is_canceled"]  

In [13]:
from sklearn.compose import ColumnTransformer, make_column_selector

children_country_pipeline = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"),
     make_column_selector(dtype_include=np.number)),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), make_column_selector(dtype_include=["object", "category"])),
], remainder="drop")


#children_country_pipeline = Pipeline([
#    ("imputer", ColumnTransformer(
#        transformers=[
#            ("children", SimpleImputer(strategy="mean"), ["children"]),
#            ("country", SimpleImputer(strategy="constant", fill_value="unknown"), ["country"]),
#        ],
#        remainder="passthrough",
#    ))
#])

X_train_imputed = children_country_pipeline.fit_transform(X_train)

# Our one-hot encoding decisions here are a little questionable. will look at later. the commented out section is where we were.

In [ ]:
from sklearn.base import clone

# Exclude the target to prevent target leakage.
leakage_cols = ["is_canceled"]
fold_X = [fold.drop(columns=leakage_cols, errors="ignore") for fold in time_folds]
fold_y = [fold["is_canceled"] for fold in time_folds]

# Each entry contains raw data ready for model fitting.
expanding_splits = []
for valid_idx in range(1, len(time_folds)):
    X_fit = pd.concat(fold_X[:valid_idx]).sort_index()
    y_fit = pd.concat(fold_y[:valid_idx]).sort_index()
    X_valid = fold_X[valid_idx]
    y_valid = fold_y[valid_idx]

    expanding_splits.append({
        "train_folds": tuple(range(valid_idx)),
        "valid_fold": valid_idx,
        "X_train": X_fit,
        "y_train": y_fit,
        "X_valid": X_valid,
        "y_valid": y_valid,
    })

# Fit preprocessing on past folds only, then transform the next fold.
def prepare_expanding_split(split):
    preprocessor = clone(children_country_pipeline)
    X_train_ready = preprocessor.fit_transform(split["X_train"])
    X_valid_ready = preprocessor.transform(split["X_valid"])

    return {
        **split,
        "preprocessor": preprocessor,
        "X_train_ready": X_train_ready,
        "X_valid_ready": X_valid_ready,
    }

# Example: prepare the first split (fold 1 predicts fold 2).
first_split = prepare_expanding_split(expanding_splits[0])

print(f"Training rows: {len(first_split['y_train']):,}")
print(f"Validation rows: {len(first_split['y_valid']):,}")
print(f"Prepared feature shapes: {first_split['X_train_ready'].shape}, "
      f"{first_split['X_valid_ready'].shape}")

Training rows: 4,295
Validation rows: 5,645
Prepared feature shapes: (4295, 117), (5645, 117)


In [15]:
from sklearn.ensemble import RandomForestClassifier

OPTIMAL_THRESHOLD = 150 / (150 + 200)  # ~0.4286

results = []

for split in expanding_splits:
    prepared = prepare_expanding_split(split)
    y_fit = prepared["y_train"]
    y_valid_fold = prepared["y_valid"].to_numpy()

    models = {
            "Logistic Regression": LogisticRegression(max_iter=1000),
            "Random Forest": RandomForestClassifier(
                n_estimators=50,
                min_samples_leaf=5,
                random_state=42,
                n_jobs=-1,
            ),
        }

    # True Baseline: Accept all bookings, paying the -$200 penalty on cancellations
    actual_kept = (y_valid_fold == 0).sum()
    actual_canceled = (y_valid_fold == 1).sum()
    baseline_revenue = 150 * actual_kept - 200 * actual_canceled

    row = {
        "Validation fold": split["valid_fold"] + 1,
        "Validation rows": len(y_valid_fold),
        "Baseline revenue ($)": baseline_revenue,
    }

    for name, model in models.items():
        model.fit(prepared["X_train_ready"], y_fit)
        
        # Predict cancellation probability and apply the cost-optimal threshold
        cancel_probs = model.predict_proba(prepared["X_valid_ready"])[:, 1]
        predictions = (cancel_probs > OPTIMAL_THRESHOLD).astype(int)

        # Target 1 means canceled. Predicting 0 means accepting the booking.
        accepted_and_kept = (predictions == 0) & (y_valid_fold == 0)
        accepted_but_canceled = (predictions == 0) & (y_valid_fold == 1)

        revenue = 150 * accepted_and_kept.sum() - 200 * accepted_but_canceled.sum()
        row[f"{name} revenue ($)"] = revenue

    results.append(row)

revenue_comparison = pd.DataFrame(results).set_index("Validation fold")
display(revenue_comparison.round(2))

c:\Users\Owner\OneDrive - University of Wisconsin-La Crosse\Documents\stat764-fall2026-fresh\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\Owner\OneDrive - University of Wisconsin-La Crosse\Documents\stat764-fall2026-fresh\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the conve

,Validation rows,Baseline revenue ($),Logistic Regression revenue ($),Random Forest revenue ($)
Validation fold,,,,
2,5645,495700,581100,594500
3,3828,344600,409200,415050
4,5597,473450,482100,503800
5,7995,473700,529550,568200
6,7631,438350,511200,635400
7,8650,386800,521050,536450
8,8170,451650,560900,572050
9,6177,295150,378300,400600
10,7379,438350,504550,553600
